# KoELECTRA 분류 모델 재학습 (Google Colab)

lotte-insight 프로젝트 — `training/train/train_classifier.py` Colab 실행용 노트북

**모델:** `monologg/koelectra-small-v3-discriminator` fine-tuning  
**분류 방식:** 멀티라벨 (BCEWithLogitsLoss + per-label pos_weight)  
**라벨:** INJURY_ROSTER / TRANSACTION_CONTRACT / MATCH_RELATED / PERFORMANCE_ANALYSIS / INTERVIEW / CLUB_OPERATION / ETC  
**입력:** title + description_snippet + game_context (경기정보)  
**목표:** val macro-F1 ≥ 0.70

---

**사전 준비 — 로컬 `lotte-insight/training/` 에서 실행**

```bash
cd lotte-insight/training

# 1. 2026 시즌 경기 결과 갱신 (필수)
python -m collect.collect_game_results --since 2026-03-01

# 2. labeled_players.csv game_context 채우기 (필수)
python -m collect.add_summaries --dataset players

# 3. 최신 팀 기사 추가 수집 (선택)
python -m collect.collect_for_labeling --days 30 --count 300 --focus-labels CLUB_OPERATION,INTERVIEW

# 4. 최신 선수 기사 추가 수집 (선택)
python -m collect.collect_players --days 30
```

> ⚠️ `python collect/xxx.py` 형태로 실행하면 `settings` 모듈을 찾지 못합니다.  
> 반드시 `python -m collect.xxx` 형태로 실행하세요 (`training/`이 sys.path에 추가됩니다).

**Colab 업로드 파일 (Cell 2)**
```
training/data/labeled_titles.csv
training/data/labeled_players.csv
training/data/game_results.csv
```

In [ ]:
# 1. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[WARN] GPU not available — 런타임 유형을 T4 GPU로 변경하세요')

In [ ]:
# 2. 학습 데이터 업로드 (로컬 → Colab)
# 파일 선택 창에서 아래 3개 파일을 선택하시오:
#   - labeled_titles.csv
#   - labeled_players.csv
#   - game_results.csv
import os
import pandas as pd
from google.colab import files

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()

for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장 완료: {dst}  ({len(content):,} bytes)')

# 업로드 확인
required = ['labeled_titles.csv', 'labeled_players.csv', 'game_results.csv']
for fname in required:
    path = f'{DATA_DIR}/{fname}'
    if os.path.exists(path):
        df = pd.read_csv(path, encoding='utf-8-sig')
        print(f'  [OK] {fname}: {len(df)}행')
    else:
        print(f'  [MISSING] {fname} — 업로드 필요')

In [ ]:
# 3. 레포 클론 및 의존성 설치
GITHUB_REPO_URL = 'https://github.com/JoeYunHa/Lotte_Insight.git'

import os, shutil, subprocess

repo_dir = '/content/lotte-insight'
is_valid_git = subprocess.run(
    ['git', '-C', repo_dir, 'status'],
    capture_output=True
).returncode == 0

if is_valid_git:
    !git -C {repo_dir} pull
else:
    if os.path.exists(repo_dir):
        shutil.rmtree(repo_dir)
        print('기존 디렉토리 제거 후 재clone')
    !git clone {GITHUB_REPO_URL} {repo_dir}

# training/data/에 업로드한 파일 복사
os.makedirs(f'{repo_dir}/training/data', exist_ok=True)
for fname in ['labeled_titles.csv', 'labeled_players.csv', 'game_results.csv']:
    src = f'/content/data/{fname}'
    dst = f'{repo_dir}/training/data/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'  복사 완료: {fname}')

!pip install -q -r {repo_dir}/training/requirements.txt
print('설치 완료')

In [ ]:
# 4. game_context 검증 (로컬에서 이미 채워진 상태 확인)
import pandas as pd

for fname, path in [
    ('labeled_titles.csv',  '/content/lotte-insight/training/data/labeled_titles.csv'),
    ('labeled_players.csv', '/content/lotte-insight/training/data/labeled_players.csv'),
]:
    df = pd.read_csv(path, encoding='utf-8-sig')
    gc = df['game_context'].fillna('').astype(str).str.strip()
    real = int(((gc != '') & (gc != '해당 날짜 경기 없음')).sum())
    print(f'{fname}: 실 경기정보 포함 {real}/{len(df)}')

In [ ]:
# 5. 학습 데이터 분포 확인
import pandas as pd
from collections import Counter

LABELS = ['MATCH_RELATED','INJURY_ROSTER','TRANSACTION_CONTRACT',
          'PERFORMANCE_ANALYSIS','INTERVIEW','CLUB_OPERATION','ETC']

total_dist = Counter()
grand_total = 0
total_with_ctx = 0

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'data/{fname}'
    df = pd.read_csv(path, encoding='utf-8-sig')
    lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
    total_dist.update(Counter(lotte['primary_label'].dropna()))
    grand_total += len(lotte)
    gc = lotte['game_context'].fillna('').astype(str).str.strip()
    has_ctx = int(((gc != '') & (gc != '해당 날짜 경기 없음')).sum())
    total_with_ctx += has_ctx
    print(f'{fname}: {len(lotte)}행  (경기정보 포함: {has_ctx}행)')

print(f'\n합산 학습 가능 행: {grand_total}')
print(f'경기정보 포함 행: {total_with_ctx} ({total_with_ctx/grand_total*100:.0f}%)')
print('\n라벨 분포:')
for label in LABELS:
    count = total_dist.get(label, 0)
    bar = '█' * (count // 100)
    print(f'  {label:<25} {count:>5}  {bar}')

minority = [l for l in LABELS if total_dist.get(l, 0) < 100]
if minority:
    print(f'\n[주의] 100건 미만 라벨: {minority} — F1 불안정 가능')

import os
if os.path.exists('data/game_results.csv'):
    gr = pd.read_csv('data/game_results.csv', encoding='utf-8-sig')
    wins = (gr['result'] == 'W').sum()
    losses = (gr['result'] == 'L').sum()
    print(f'\ngame_results.csv: {len(gr)}경기 (W:{wins} L:{losses})')
    print(f'  기간: {gr["game_date"].min()} ~ {gr["game_date"].max()}')

In [ ]:
# 6. 학습 실행 — T4 GPU 기준 약 8~15분
!cd /content/lotte-insight/training && \
  PYTHONPATH=/content/lotte-insight/training \
  python train/train_classifier.py \
    --epochs 5 \
    --lr 5e-5 \
    --batch 16

In [ ]:
# 7. 학습 결과 확인
import json, os

threshold_path = '/content/lotte-insight/training/models/classifier_koelectra/label_thresholds.json'
if os.path.exists(threshold_path):
    with open(threshold_path) as f:
        thresholds = json.load(f)
    print('Per-label 최적 임계값:')
    for label, t in thresholds.items():
        print(f'  {label:<25} {t:.2f}')
else:
    print('[WARN] label_thresholds.json 없음')

print()
!cd /content/lotte-insight/training && \
  PYTHONPATH=/content/lotte-insight/training \
  python train/train_classifier.py --eval-only

# macro-F1 < 0.70 이면 아래 주석 해제
# !cd /content/lotte-insight/training && \
#   PYTHONPATH=/content/lotte-insight/training \
#   python train/train_classifier.py --epochs 8 --lr 3e-5 --batch 16

In [ ]:
# 8. 학습된 모델 다운로드 (Colab → 로컬)
import shutil, os
from google.colab import files

MODEL_DIR = '/content/lotte-insight/training/models/classifier_koelectra'
ZIP_PATH = '/content/classifier_koelectra.zip'

if os.path.exists(MODEL_DIR):
    shutil.make_archive('/content/classifier_koelectra', 'zip', MODEL_DIR)
    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f'압축 완료: {ZIP_PATH} ({size_mb:.1f} MB)')
    files.download(ZIP_PATH)
else:
    # 실제 저장 위치 탐색
    import glob
    found = glob.glob('/content/**/classifier_koelectra', recursive=True)
    if found:
        print(f'모델 발견: {found[0]}')
        shutil.make_archive('/content/classifier_koelectra', 'zip', found[0])
        files.download(ZIP_PATH)
    else:
        print('[ERROR] 모델을 찾을 수 없음')
        print('저장된 모델 디렉토리 목록:')
        !find /content -name "label_encoder.json" 2>/dev/null

In [ ]:
# 9. 추론 테스트 (학습/추론 일치 검증)
import json, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = 'models/classifier_koelectra'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

with open(f'{MODEL_DIR}/label_encoder.json', encoding='utf-8') as f:
    LABELS = json.load(f)
with open(f'{MODEL_DIR}/label_thresholds.json', encoding='utf-8') as f:
    THRESHOLDS = json.load(f)

def predict(title: str, description: str = '', game_context: str = '') -> list[str]:
    """max_length=128 — backend/models/classifier.py와 동일 설정."""
    auxiliary = description[:300].strip()
    ctx = str(game_context or '').strip()
    if ctx and ctx != '해당 날짜 경기 없음':
        auxiliary = f'{auxiliary} [경기정보] 경기: {ctx}' if auxiliary else f'경기: {ctx}'
    enc = tokenizer(
        title, auxiliary,
        truncation='only_second', padding='max_length',
        max_length=128,
        return_tensors='pt',
    )
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits[0])
    result = [LABELS[i] for i, p in enumerate(probs) if p.item() >= THRESHOLDS.get(LABELS[i], 0.5)]
    return result if result else ['ETC']

SAMPLES = [
    ('롯데 나균안, 시즌 5승 달성…선발 로테이션 안정화',
     '나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 따냈다.',
     '롯데 vs 두산 (홈) 7-3 승 | 결승타: 나균안(6회) | 타자 키플레이어: 전준우(롯데)'),
    ('롯데 전준우 햄스트링 부상…2주 결장 예상',
     '전준우가 경기 중 부상으로 1군 엔트리에서 말소됐다.', ''),
    ('롯데 구단, 외국인 투수 교체 결정',
     '롯데가 부진한 외국인 투수를 방출하고 새 용병을 물색 중이다.', ''),
    ('서튼 감독 "선수들이 잘 따라줬다"',
     '래리 서튼 감독이 경기 후 인터뷰에서 선수단을 칭찬했다.',
     '롯데 vs KIA (원정) 4-2 승'),
    ('사직구장 개막 이벤트, 팬 3만 명 몰려',
     '롯데 자이언츠가 홈 개막전 기념 이벤트를 성황리에 개최했다.', ''),
]

print('추론 결과 (max_length=128, game_context 포함):')
for title, desc, ctx in SAMPLES:
    predicted = predict(title, desc, ctx)
    ctx_note = '[경기정보 있음]' if ctx else ''
    print(f'  [{", ".join(predicted)}] {title[:40]} {ctx_note}')